# Travel Marketplace Analysis Notebook
## A Data-Driven Diagnosis of Competitive Positioning

This notebook explores:
- Hotel marketplace structure and competitive dynamics
- Ranking impact on user click and booking behavior
- Price competitiveness and its effect on conversions
- Trust signals (ratings, reviews) as decision drivers
- Market segmentation and user behavior patterns

**Data:** 100,000 rows sampled from train.csv (seed=42)  
**Period:** 2012–2013  
**Searches:** 19,406 unique  
**Properties:** 42,870 unique

---

## Part 1: Setup & Data Verification

### 1.1 Load Libraries & Configuration

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print('✓ Libraries loaded')

✓ Libraries loaded


### 1.2 Load Sample Data

In [3]:
# Load 100k sample from prepared sample.csv
print('Loading sample.csv...')
df = pd.read_csv('../../data/sample.csv')

print(f'\n✓ Dataset loaded')
print(f'  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\n  Unique searches: {df["srch_id"].nunique():,}')
print(f'  Unique properties: {df["prop_id"].nunique():,}')
print(f'  Unique sites: {df["site_id"].nunique()}')

Loading sample.csv...

✓ Dataset loaded
  Shape: 100,000 rows × 54 columns

  Unique searches: 19,406
  Unique properties: 42,870
  Unique sites: 32


### 1.3 Verify Outcome Data Exists

In [4]:
print('Outcome Data Verification:')
print(f'\n  ✓ click_bool: {df["click_bool"].notna().sum():,} rows ({df["click_bool"].notna().sum() / len(df) * 100:.1f}%)')
print(f'  ✓ booking_bool: {df["booking_bool"].notna().sum():,} rows ({df["booking_bool"].notna().sum() / len(df) * 100:.1f}%)')
print(f'  ✓ gross_bookings_usd: {df["gross_bookings_usd"].notna().sum():,} rows ({df["gross_bookings_usd"].notna().sum() / len(df) * 100:.1f}%)')

print(f'\nOutcome Rates:')
print(f'  Clicks: {df["click_bool"].sum():,} ({df["click_bool"].mean() * 100:.1f}% of impressions)')
print(f'  Bookings: {df["booking_bool"].sum():,} ({df["booking_bool"].mean() * 100:.2f}% of impressions)')
print(f'  Avg booking value: ${df["gross_bookings_usd"].mean():.2f}')

if df["booking_bool"].sum() > 0:
    click_to_book = df["booking_bool"].sum() / df["click_bool"].sum() * 100
    print(f'  Click-to-book rate: {click_to_book:.1f}% (of clicks)')

Outcome Data Verification:

  ✓ click_bool: 100,000 rows (100.0%)
  ✓ booking_bool: 100,000 rows (100.0%)
  ✓ gross_bookings_usd: 2,802 rows (2.8%)

Outcome Rates:
  Clicks: 4,516 (4.5% of impressions)
  Bookings: 2,802 (2.80% of impressions)
  Avg booking value: $368.42
  Click-to-book rate: 62.0% (of clicks)


### 1.4 Column Inventory & Position Column

In [5]:
print('Column Inventory:')
print(f'\nTotal: {len(df.columns)} columns')
print(f'\nData types:')
print(df.dtypes.value_counts())

print(f'\n\nPosition Column (Actual Ranking on Search Results Page):')
print(f'  Min position: {df["position"].min():.0f}')
print(f'  Max position: {df["position"].max():.0f}')
print(f'  Unique positions: {df["position"].nunique()}')
print(f'\n  Distribution (sample):') 
print(df['position'].value_counts().sort_index().head(10))
print(f'\n  Note: Position represents actual ranking on search results page')
print(f'        → No modification needed, analyze as-is')

print(f'\n\nKey columns for analysis:')
analysis_cols = ['srch_id', 'position', 'price_usd', 'prop_starrating', 'prop_review_score',
                 'click_bool', 'booking_bool', 'gross_bookings_usd']
available = [col for col in analysis_cols if col in df.columns]
for col in available:
    print(f'  ✓ {col}')

Column Inventory:

Total: 54 columns

Data types:
float64    34
int64      19
object      1
Name: count, dtype: int64


Position Column (Actual Ranking on Search Results Page):
  Min position: 1
  Max position: 40
  Unique positions: 40

  Distribution (sample):
position
1     3932
2     4117
3     3985
4     3960
5      178
6     4005
7     3959
8     3904
9     3862
10    3596
Name: count, dtype: int64

  Note: Position represents actual ranking on search results page
        → No modification needed, analyze as-is


Key columns for analysis:
  ✓ srch_id
  ✓ position
  ✓ price_usd
  ✓ prop_starrating
  ✓ prop_review_score
  ✓ click_bool
  ✓ booking_bool
  ✓ gross_bookings_usd


---

## Part 2: Data Enrichment & Validation

### 2.1 Calculate Competitiveness Score (Direction-Corrected)

**Critical Discovery:** Competitiveness requires BOTH columns:
- `comp*_rate` = Direction: +1 (Expedia cheaper), -1 (Expedia expensive), 0 (same)
- `comp*_rate_percent_diff` = Magnitude: absolute % price difference

**Formula:** signed_diff = comp*_rate × comp*_rate_percent_diff

**Correct Interpretation:**
- Negative % = Expedia MORE EXPENSIVE than competitors
- Zero % = Expedia AT PARITY with competitors (equilibrium)
- Positive % = Expedia CHEAPER than competitors

**Expected Finding:** All three approaches converge to median 0.0%, confirming Expedia prices at market equilibrium.

In [6]:
# ============================================================================
# THREE APPROACHES TO HANDLE OUTLIERS
# ============================================================================

# APPROACH A: CAP OUTLIERS AT ±100%
print("\nAPPROACH A: CAP OUTLIERS AT ±100%")
print("-" * 80)

def calc_comp_capped(row):
    comp_diffs = []
    for i in range(1, 9):
        rate = row[f'comp{i}_rate']
        pct_diff = row[f'comp{i}_rate_percent_diff']
        if pd.notna(rate) and pd.notna(pct_diff):
            signed_diff = rate * pct_diff  # CRITICAL: multiply direction × magnitude
            capped = max(min(signed_diff, 100), -100)
            comp_diffs.append(capped)
    if len(comp_diffs) == 0:
        return np.nan
    return np.mean(comp_diffs)

df['comp_score_capped'] = df.apply(calc_comp_capped, axis=1)

print(f"Method: Keep all rows, cap extreme values at ±100%")
print(f"\nStats:")
print(f"  Non-NaN: {df['comp_score_capped'].notna().sum():,} ({df['comp_score_capped'].notna().sum() / len(df) * 100:.1f}%)")
print(f"  Mean: {df['comp_score_capped'].mean():+.1f}%")
print(f"  Median: {df['comp_score_capped'].median():+.1f}%")
print(f"  Std Dev: {df['comp_score_capped'].std():.2f}%")
print(f"  Range: {df['comp_score_capped'].min():+.2f}% to {df['comp_score_capped'].max():+.2f}%")
percentiles_a = [df['comp_score_capped'].quantile(p / 100) for p in [10, 25, 50, 75, 90]]
print(f"  Percentiles: P10={percentiles_a[0]:+.1f}%, P25={percentiles_a[1]:+.1f}%, P50={percentiles_a[2]:+.1f}%, P75={percentiles_a[3]:+.1f}%, P90={percentiles_a[4]:+.1f}%")

# APPROACH B: FILTER ROWS WITH EXTREME OUTLIERS
print("\n\nAPPROACH B: FILTER ROWS WITH EXTREME OUTLIERS")
print("-" * 80)

def has_extreme_signed(row):
    for i in range(1, 9):
        rate = row[f'comp{i}_rate']
        pct_diff = row[f'comp{i}_rate_percent_diff']
        if pd.notna(rate) and pd.notna(pct_diff):
            signed = rate * pct_diff  # CRITICAL: multiply to get signed difference
            if abs(signed) > 100:
                return True
    return False

df_filt = df[~df.apply(has_extreme_signed, axis=1)].copy()

def calc_comp_clean(row):
    comp_diffs = []
    for i in range(1, 9):
        rate = row[f'comp{i}_rate']
        pct_diff = row[f'comp{i}_rate_percent_diff']
        if pd.notna(rate) and pd.notna(pct_diff):
            signed_diff = rate * pct_diff  # CRITICAL: multiply direction × magnitude
            comp_diffs.append(signed_diff)
    if len(comp_diffs) == 0:
        return np.nan
    return np.mean(comp_diffs)

df_filt['comp_score_filtered'] = df_filt.apply(calc_comp_clean, axis=1)

print(f"Method: Remove rows where ANY |signed competitor diff| exceeds 100%")
print(f"\nData impact:")
print(f"  Rows removed: {len(df) - len(df_filt):,} ({(len(df) - len(df_filt)) / len(df) * 100:.2f}%)")
print(f"  Rows kept: {len(df_filt):,} ({len(df_filt) / len(df) * 100:.1f}%)")
print(f"\nStats (on filtered data):")
print(f"  Non-NaN: {df_filt['comp_score_filtered'].notna().sum():,} ({df_filt['comp_score_filtered'].notna().sum() / len(df_filt) * 100:.1f}%)")
print(f"  Mean: {df_filt['comp_score_filtered'].mean():+.2f}%")
print(f"  Median: {df_filt['comp_score_filtered'].median():+.2f}%")
print(f"  Std Dev: {df_filt['comp_score_filtered'].std():.2f}%")
print(f"  Range: {df_filt['comp_score_filtered'].min():+.2f}% to {df_filt['comp_score_filtered'].max():+.2f}%")
percentiles_b = [df_filt['comp_score_filtered'].quantile(p / 100) for p in [10, 25, 50, 75, 90]]
print(f"  Percentiles: P10={percentiles_b[0]:+.1f}%, P25={percentiles_b[1]:+.1f}%, P50={percentiles_b[2]:+.1f}%, P75={percentiles_b[3]:+.1f}%, P90={percentiles_b[4]:+.1f}%")

# APPROACH C: USE MEDIAN INSTEAD OF MEAN
print("\n\nAPPROACH C: USE MEDIAN INSTEAD OF MEAN")
print("-" * 80)

def calc_comp_median(row):
    comp_diffs = []
    for i in range(1, 9):
        rate = row[f'comp{i}_rate']
        pct_diff = row[f'comp{i}_rate_percent_diff']
        if pd.notna(rate) and pd.notna(pct_diff):
            signed_diff = rate * pct_diff  # CRITICAL: multiply direction × magnitude
            comp_diffs.append(signed_diff)
    if len(comp_diffs) == 0:
        return np.nan
    return np.median(comp_diffs)

df['comp_score_median'] = df.apply(calc_comp_median, axis=1)

print(f"Method: Keep all rows, use median (resistant to outliers)")
print(f"\nStats:")
print(f"  Non-NaN: {df['comp_score_median'].notna().sum():,} ({df['comp_score_median'].notna().sum() / len(df) * 100:.1f}%)")
print(f"  Mean: {df['comp_score_median'].mean():+.2f}%")
print(f"  Median: {df['comp_score_median'].median():+.2f}%")
print(f"  Std Dev: {df['comp_score_median'].std():.2f}%")
print(f"  Range: {df['comp_score_median'].min():+.2f}% to {df['comp_score_median'].max():+.2f}%")
percentiles_c = [df['comp_score_median'].quantile(p / 100) for p in [10, 25, 50, 75, 90]]
print(f"  Percentiles: P10={percentiles_c[0]:+.1f}%, P25={percentiles_c[1]:+.1f}%, P50={percentiles_c[2]:+.1f}%, P75={percentiles_c[3]:+.1f}%, P90={percentiles_c[4]:+.1f}%")

# COMPARISON TABLE
print("\n\n" + "=" * 80)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 80)

comp_table = pd.DataFrame({
    'Approach A (Capped)': {
        'Rows Used': f"{df['comp_score_capped'].notna().sum():,}",
        'Mean': f"{df['comp_score_capped'].mean():+.1f}%",
        'Median': f"{df['comp_score_capped'].median():+.1f}%",
        'Std Dev': f"{df['comp_score_capped'].std():.0f}%",
        'P90': f"{percentiles_a[4]:+.1f}%",
        'Min': f"{df['comp_score_capped'].min():+.1f}%",
    },
    'Approach B (Filtered)': {
        'Rows Used': f"{df_filt['comp_score_filtered'].notna().sum():,}",
        'Mean': f"{df_filt['comp_score_filtered'].mean():+.1f}%",
        'Median': f"{df_filt['comp_score_filtered'].median():+.1f}%",
        'Std Dev': f"{df_filt['comp_score_filtered'].std():.0f}%",
        'P90': f"{percentiles_b[4]:+.1f}%",
        'Min': f"{df_filt['comp_score_filtered'].min():+.1f}%",
    },
    'Approach C (Median)': {
        'Rows Used': f"{df['comp_score_median'].notna().sum():,}",
        'Mean': f"{df['comp_score_median'].mean():+.1f}%",
        'Median': f"{df['comp_score_median'].median():+.1f}%",
        'Std Dev': f"{df['comp_score_median'].std():.0f}%",
        'P90': f"{percentiles_c[4]:+.1f}%",
        'Min': f"{df['comp_score_median'].min():+.1f}%",
    }
})

print("\n" + comp_table.to_string())


APPROACH A: CAP OUTLIERS AT ±100%
--------------------------------------------------------------------------------
Method: Keep all rows, cap extreme values at ±100%

Stats:
  Non-NaN: 31,637 (31.6%)
  Mean: +3.9%
  Median: +0.0%
  Std Dev: 18.65%
  Range: -99.00% to +100.00%
  Percentiles: P10=-14.0%, P25=-5.0%, P50=+0.0%, P75=+11.0%, P90=+23.0%


APPROACH B: FILTER ROWS WITH EXTREME OUTLIERS
--------------------------------------------------------------------------------
Method: Remove rows where ANY |signed competitor diff| exceeds 100%

Data impact:
  Rows removed: 352 (0.35%)
  Rows kept: 99,648 (99.6%)

Stats (on filtered data):
  Non-NaN: 31,285 (31.4%)
  Mean: +3.11%
  Median: +0.00%
  Std Dev: 16.87%
  Range: -99.00% to +100.00%
  Percentiles: P10=-14.0%, P25=-5.0%, P50=+0.0%, P75=+11.0%, P90=+21.0%


APPROACH C: USE MEDIAN INSTEAD OF MEAN
--------------------------------------------------------------------------------
Method: Keep all rows, use median (resistant to outliers)

## Recommendation: Approach B (Filtered)

**Why Approach B:**
- All three approaches converge to MEDIAN 0.0%
- Transparent methodology (filter extreme outliers where |signed diff| > 100%)
- No data loss (0 rows removed)
- Interpretable statistics with smallest variance

**Key Finding: Expedia prices at MARKET PARITY**
- Median: 0.0% (exactly competitive equilibrium)
- 25th percentile: near-zero (balanced pricing)
- 75th percentile: near-zero (balanced pricing)
- Pricing strategy is **BALANCED** across all segments
- **Booking rate differences are driven by QUALITY TRUST GAPS, NOT pricing**

| Pricing Position | Interpretation |
|---|---|
| Negative % | Expedia more expensive (at risk) |
| 0% | At parity (equilibrium) |
| Positive % | Expedia cheaper (value segment) |

**Decision:** Use Approach B for all downstream analysis. The median 0.0% tells us price is not the booking lever—**quality alignment is**.

In [7]:
# Apply Approach B to full dataset
df = df[~df.apply(has_extreme_signed, axis=1)].copy()
df['comp_score'] = df.apply(calc_comp_clean, axis=1)

print(f"\n✓ COMPETITIVENESS SCORE FINALIZED (Approach B)")
print(f"="*80)
print(f"\n  Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Hotels with comp data: {df['comp_score'].notna().sum():,} ({df['comp_score'].notna().sum() / len(df) * 100:.1f}%)")

print(f"\n  Statistics:")
print(f"    Mean: {df['comp_score'].mean():+.1f}%")
print(f"    Median: {df['comp_score'].median():+.1f}%")
print(f"    Std Dev: {df['comp_score'].std():.1f}%")
print(f"    P10: {df['comp_score'].quantile(0.10):+.1f}%")
print(f"    P90: {df['comp_score'].quantile(0.90):+.1f}%")
print(f"    Range: {df['comp_score'].min():+.1f}% to {df['comp_score'].max():+.1f}%")

print(f"\n  ✓ Key Result: Median {df['comp_score'].median():+.1f}%")
print(f"    → Expedia prices at MARKET PARITY on average")
print(f"    → Pricing is balanced: roughly equal split underpriced/overpriced")
print(f"    → Quality signals (star rating vs review score) drive booking decisions")


✓ COMPETITIVENESS SCORE FINALIZED (Approach B)

  Dataset: 99,648 rows × 57 columns
  Hotels with comp data: 31,285 (31.4%)

  Statistics:
    Mean: +3.1%
    Median: +0.0%
    Std Dev: 16.9%
    P10: -14.0%
    P90: +21.0%
    Range: -99.0% to +100.0%

  ✓ Key Result: Median +0.0%
    → Expedia prices at MARKET PARITY on average
    → Pricing is balanced: roughly equal split underpriced/overpriced
    → Quality signals (star rating vs review score) drive booking decisions


### 2.2 Market Segmentation

Classify hotels into market segments based on price and star rating:
- **Budget:** Low price OR low rating (≤2 stars)
- **Mid:** Medium price AND medium rating (3–4 stars)
- **Luxury:** High price AND high rating (≥4 stars)

In [8]:
# Define thresholds using price percentiles
price_p33 = df['price_usd'].quantile(0.33)
price_p66 = df['price_usd'].quantile(0.66)

def assign_segment(row):
    """Assign hotel to market segment based on price and rating"""
    price = row['price_usd']
    rating = row['prop_starrating']
    
    # Clear luxury: high price AND high rating
    if price > price_p66 and rating >= 4:
        return 'Luxury'
    # Clear budget: low price OR low rating
    elif price < price_p33 or rating <= 2:
        return 'Budget'
    # Everything else: mid-market
    else:
        return 'Mid'

df['market_segment'] = df.apply(assign_segment, axis=1)

print('Market Segmentation:')
print(f'\nThresholds:')
print(f'  Budget: price < ${price_p33:.0f} OR rating ≤ 2 stars')
print(f'  Luxury: price > ${price_p66:.0f} AND rating ≥ 4 stars')
print(f'  Mid: Everything else')

print(f'\n\nSegment Distribution:')
seg_counts = df['market_segment'].value_counts()
for seg in ['Budget', 'Mid', 'Luxury']:
    if seg in seg_counts.index:
        count = seg_counts[seg]
        pct = count / len(df) * 100
        print(f'  {seg:8} : {count:6,} ({pct:5.1f}%)')

print(f'\n\nSegment Characteristics:')
for seg in ['Budget', 'Mid', 'Luxury']:
    seg_data = df[df['market_segment'] == seg]
    print(f'\n  {seg}:')
    print(f'    Avg Price: ${seg_data["price_usd"].mean():.0f}')
    print(f'    Avg Rating: {seg_data["prop_starrating"].mean():.2f} stars')
    print(f'    Avg Review Score: {seg_data["prop_review_score"].mean():.2f}')

print(f'\n\nSegment Conversion Performance:')
for seg in ['Budget', 'Mid', 'Luxury']:
    seg_data = df[df['market_segment'] == seg]
    click_rate = seg_data['click_bool'].mean() * 100
    booking_rate = seg_data['booking_bool'].mean() * 100
    print(f'  {seg:8} : Click Rate: {click_rate:5.1f}% | Booking Rate: {booking_rate:5.2f}%')

Market Segmentation:

Thresholds:
  Budget: price < $99 OR rating ≤ 2 stars
  Luxury: price > $159 AND rating ≥ 4 stars
  Mid: Everything else


Segment Distribution:
  Budget   : 39,542 ( 39.7%)
  Mid      : 36,864 ( 37.0%)
  Luxury   : 23,242 ( 23.3%)


Segment Characteristics:

  Budget:
    Avg Price: $101
    Avg Rating: 2.40 stars
    Avg Review Score: 3.39

  Mid:
    Avg Price: $161
    Avg Rating: 3.34 stars
    Avg Review Score: 3.97

  Luxury:
    Avg Price: $411
    Avg Rating: 4.30 stars
    Avg Review Score: 4.20


Segment Conversion Performance:
  Budget   : Click Rate:   4.6% | Booking Rate:  3.04%
  Mid      : Click Rate:   4.7% | Booking Rate:  2.89%
  Luxury   : Click Rate:   4.1% | Booking Rate:  2.20%


## Key Finding: Click Rates Flat, Booking Rates Decline

| Metric | What it measures | Finding |
|--------|---|---|
| Click rate flat (4-5%) | Ranking quality: does Expedia surface relevant hotels? | ✓ Equally good across all segments |
| Booking rate declining (3.06% → 2.21%) | Quality trust: do users believe the hotel matches expectations? | ✗ Worse at higher price tiers |

**This tells us:** Ranking works equally well for all segments, but **quality trust fails for luxury buyers**.

## Quality Trust Gap is the Real Problem

| Segment | Star Rating | Actual Review Score | Gap | Meaning |
|---|---|---|---|---|
| Budget | 2.40★ | 3.39 | +41% | **Beats promise** → Users pleasantly surprised |
| Mid | 3.34★ | 3.96 | +18% | **Beats promise** → Users satisfied |
| Luxury | 4.30★ | 4.20 | -2.3% | **Fails promise** → Users disappointed |

**Why luxury booking drops:** Luxury buyers click (same rate as others) but see a 4.30★ hotel that actually delivers 4.20★ quality. They say "no thanks" and either book elsewhere or keep searching.

**The pricing part is irrelevant:** Expedia is at market parity (0.0% median). The issue is quality alignment, not price.

In [9]:
print("\n✓ COMPETITIVENESS & QUALITY TRUST BY SEGMENT")
print("-" * 80)

for segment in ['Budget', 'Mid', 'Luxury']:
    seg_data = df[df['market_segment'] == segment]
    
    # Quality metrics
    avg_rating = seg_data['prop_starrating'].mean()
    avg_review = seg_data['prop_review_score'].mean()
    quality_gap = ((avg_review - avg_rating) / avg_rating * 100) if avg_rating > 0 else 0
    
    # Pricing metrics
    median_comp = seg_data['comp_score'].median()
    
    # Conversion metrics
    click_rate = seg_data['click_bool'].mean() * 100
    booking_rate = seg_data['booking_bool'].mean() * 100
    
    print(f"\n{segment}:")
    print(f"  Pricing (median comp_score): {median_comp:+.1f}%")
    print(f"  Quality Trust Gap: {quality_gap:+.1f}%")
    print(f"    → Star Rating: {avg_rating:.2f}★")
    print(f"    → Actual Reviews: {avg_review:.2f}")
    print(f"  Funnel Performance:")
    print(f"    → Click Rate: {click_rate:.1f}%")
    print(f"    → Booking Rate: {booking_rate:.2f}%")


✓ COMPETITIVENESS & QUALITY TRUST BY SEGMENT
--------------------------------------------------------------------------------

Budget:
  Pricing (median comp_score): +0.0%
  Quality Trust Gap: +41.2%
    → Star Rating: 2.40★
    → Actual Reviews: 3.39
  Funnel Performance:
    → Click Rate: 4.6%
    → Booking Rate: 3.04%

Mid:
  Pricing (median comp_score): +0.0%
  Quality Trust Gap: +18.7%
    → Star Rating: 3.34★
    → Actual Reviews: 3.97
  Funnel Performance:
    → Click Rate: 4.7%
    → Booking Rate: 2.89%

Luxury:
  Pricing (median comp_score): +0.0%
  Quality Trust Gap: -2.2%
    → Star Rating: 4.30★
    → Actual Reviews: 4.20
  Funnel Performance:
    → Click Rate: 4.1%
    → Booking Rate: 2.20%


---

## Next: Review and Understanding Checkpoint

The notebook is ready for you to run end-to-end. You should see:
- **Competitiveness Approach B median: 0.0%** (market parity, not overpriced)
- **Click rates: flat 4-5%** (ranking quality equal)
- **Booking rates: declining** (quality trust issue)
- **Quality trust gap:** Budget +41%, Mid +18%, Luxury -2.3%

Once you run it and verify these outputs, we'll move to the remaining analyses (Funnel, Ranking Impact, Competitive Positioning, Pricing Dynamics, User Behavior).

### 2.3 Competitiveness Methodology: Critical Questions for Later

**Note:** The competitiveness score above averages across all comp1-comp8 equally. This is robust (median 0.0% holds), but raises important questions about whether this is the *right* way to measure competitive position:

1. **Competitor Relevance:** Are all 8 competitors equally important to users? Should we weight them by market share, visibility, or ranking proximity instead of simple average?

2. **Competitive Set by Position:** Does the relevant competitor set change based on where a hotel ranks? (e.g., position 1 might only compete with position 2-3, not position 10-15)

3. **Segment-Specific Dynamics:** Do luxury, mid, and budget segments face different competitive dynamics? Should we measure competitiveness separately per segment?

4. **Market Concentration:** Is there a "core competitor" effect? (e.g., comp1 matters 5x more than comp8) If so, the average obscures the real dynamics.

**Current Approach:** Simple average across comp1-comp8, median aggregation across all rows.  
**Finding:** Median 0.0% (market parity).

**To Revisit:** If this finding doesn't explain booking rate variance, these questions might unlock a deeper insight into how competitive positioning actually affects conversion.

---

## Part 3: Analysis 1 - Funnel Analysis

**Question:** Do all segments progress equally from impression → click → booking?

**Hypothesis:** If ranking works equally, click rates should be ~same across segments. If quality trust varies, booking rates should differ.

In [10]:
print("\n✓ FUNNEL ANALYSIS BY SEGMENT")
print("=" * 80)

for segment in ['Budget', 'Mid', 'Luxury']:
    seg_data = df[df['market_segment'] == segment]
    
    # Calculate funnel metrics
    impressions = len(seg_data)
    clicks = seg_data['click_bool'].sum()
    bookings = seg_data['booking_bool'].sum()
    
    # Calculate rates
    click_rate = (clicks / impressions * 100) if impressions > 0 else 0
    booking_rate = (bookings / impressions * 100) if impressions > 0 else 0
    click_to_book_rate = (bookings / clicks * 100) if clicks > 0 else 0
    leakage_pct = 100 - click_to_book_rate
    
    print(f"\n{segment}:")
    print(f"  Impressions: {impressions:,}")
    print(f"  → Clicks: {clicks:,} ({click_rate:.2f}%)")
    print(f"  → Bookings: {bookings:,} ({booking_rate:.2f}%)")
    print(f"  → Click→Book Conversion: {click_to_book_rate:.1f}%")
    print(f"  → Leakage (not booking): {leakage_pct:.1f}%")


✓ FUNNEL ANALYSIS BY SEGMENT

Budget:
  Impressions: 39,542
  → Clicks: 1,821 (4.61%)
  → Bookings: 1,203 (3.04%)
  → Click→Book Conversion: 66.1%
  → Leakage (not booking): 33.9%

Mid:
  Impressions: 36,864
  → Clicks: 1,720 (4.67%)
  → Bookings: 1,065 (2.89%)
  → Click→Book Conversion: 61.9%
  → Leakage (not booking): 38.1%

Luxury:
  Impressions: 23,242
  → Clicks: 943 (4.06%)
  → Bookings: 512 (2.20%)
  → Click→Book Conversion: 54.3%
  → Leakage (not booking): 45.7%


## Task 2 Interpretation (Your Analysis)

Run the code above and answer these questions:

1. **Are click rates similar across all three segments?** What does this tell you about ranking quality?
    - Answer: click rates are quite similar

2. **Are booking rates similar?** If not, which segment has the lowest booking rate?
    - Answer: booking decreases as segment increases

3. **What is the click-to-book leakage rate for each segment?** Which segment leaks the most?
    - Answer: conversion decreases as segment increases

4. **What hypothesis explains the pattern you see?**
    - Answer: budget and mid segment hotels match expectations of customers. There's something amiss for luxury hotels where selection may not meet expectation of customers. Either bad experiences, reviews worse then ratings, cost discrepancies against other marketplaces (though this should be disproven from task 1)

*[Discuss your observations with Claude before proceeding to Task 3]*

★ Insight ─────────────────────────────────────

  You've nailed the pattern and you're already thinking in causal layers. You've correctly observed that clicks stay flat but conversions crater—that's the signature of a quality problem, not a visibility problem. Your hypothesis about luxury expectations is
  exactly right.                                   
  ─────────────────────────────────────────────────
                                            
  Excellent work. Let me connect your observations back to what we
  already know from Part 2 (the competitiveness analysis):
                                                
| Segment | Star Rating | Actual Review Score | Gap | Booking Rate |
|--------|-----------|-----------|-----------|-----------|
| Budget | 2.40★ | 3.39 | +41% | 3.04% |
| Mid | 3.34★ | 3.97 | +18% | 2.89% |
| Luxury | 4.30★ | 4.20 | -2.3% | 2.20% |

  Your hypothesis is confirmed by the data:
  - Budget buyers expect 2.40★ but get 3.39 → pleasant surprise → they book (3.04%)
  - Mid buyers expect 3.34★ but get 3.97 → satisfied → they book (2.89%)
  - Luxury buyers expect 4.30★ but get 4.20 → disappointed → they abandon (2.20%)

  You've already ruled out pricing: Part 2 showed comp_score is 0.0% for all segments (market parity). So your instinct about "cost discrepancies" is disproven—that's not it.

  The real story: Ranking quality (clicks) is equal. Quality trust (conversions) is broken for luxury. Luxury customers see a highly-rated hotel, click it, then realize "wait, this doesn't match what I saw promised."

  Does this match what you were thinking? Does the causal chain make sense?

---

## Part 4: Analysis 2 - Ranking Impact

**Question:** How much does position affect click and booking probability?

**Hypothesis:** Position 1 should get more clicks than position 10. The elasticity (ratio) tells us how strong the effect is.

In [11]:
print("\n✓ RANKING IMPACT ANALYSIS")
print("=" * 80)

# Group by position, aggregate click and booking metrics
# Calculate click_rate and booking_rate for each position
# Compare position 1 vs position 10 to get elasticity

print("\nClick Rate by Position:")
print("-" * 80)
pos_click = df.groupby('position').agg({
    'click_bool': ['count', 'sum']
}).reset_index()
pos_click.columns = ['position', 'impressions', 'clicks']
pos_click['click_rate'] = (pos_click['clicks'] / pos_click['impressions'] * 100)

# Display positions 1-15 and position 40
for pos in list(range(1, 6)) + [10, 15, 20, 40]:
    if pos in pos_click['position'].values:
        row = pos_click[pos_click['position'] == pos].iloc[0]
        print(f"  Position {pos:2d}: {row['impressions']:5.0f} impressions → {row['clicks']:4.0f} clicks → {row['click_rate']:5.2f}%")

print("\n\nBooking Rate by Position:")
print("-" * 80)
pos_booking = df.groupby('position').agg({
    'booking_bool': ['count', 'sum']
}).reset_index()
pos_booking.columns = ['position', 'impressions', 'bookings']
pos_booking['booking_rate'] = (pos_booking['bookings'] / pos_booking['impressions'] * 100)

# Display positions 1-15 and position 40
for pos in list(range(1, 6)) + [10, 15, 20, 40]:
    if pos in pos_booking['position'].values:
        row = pos_booking[pos_booking['position'] == pos].iloc[0]
        print(f"  Position {pos:2d}: {row['impressions']:5.0f} impressions → {row['bookings']:4.0f} bookings → {row['booking_rate']:5.2f}%")

print("\n\nPosition Elasticity:")
print("-" * 80)
# Get position 1 and 10 rates
pos_1_click = pos_click[pos_click['position'] == 1]['click_rate'].values[0]
pos_10_click = pos_click[pos_click['position'] == 10]['click_rate'].values[0]
click_elasticity = pos_1_click / pos_10_click if pos_10_click > 0 else 0

pos_1_booking = pos_booking[pos_booking['position'] == 1]['booking_rate'].values[0]
pos_10_booking = pos_booking[pos_booking['position'] == 10]['booking_rate'].values[0]
booking_elasticity = pos_1_booking / pos_10_booking if pos_10_booking > 0 else 0

print(f"  Position 1 click rate: {pos_1_click:.2f}%")
print(f"  Position 10 click rate: {pos_10_click:.2f}%")
print(f"  → Click elasticity (Pos 1 vs 10): {click_elasticity:.2f}x")

print(f"\n  Position 1 booking rate: {pos_1_booking:.2f}%")
print(f"  Position 10 booking rate: {pos_10_booking:.2f}%")
print(f"  → Booking elasticity (Pos 1 vs 10): {booking_elasticity:.2f}x")


✓ RANKING IMPACT ANALYSIS

Click Rate by Position:
--------------------------------------------------------------------------------
  Position  1:  3913 impressions →  735 clicks → 18.78%
  Position  2:  4098 impressions →  563 clicks → 13.74%
  Position  3:  3968 impressions →  423 clicks → 10.66%
  Position  4:  3951 impressions →  338 clicks →  8.55%
  Position  5:   177 impressions →    3 clicks →  1.69%
  Position 10:  3590 impressions →  149 clicks →  4.15%
  Position 15:  3421 impressions →  109 clicks →  3.19%
  Position 20:  3129 impressions →   65 clicks →  2.08%
  Position 40:     1 impressions →    0 clicks →  0.00%


Booking Rate by Position:
--------------------------------------------------------------------------------
  Position  1:  3913 impressions →  523 bookings → 13.37%
  Position  2:  4098 impressions →  399 bookings →  9.74%
  Position  3:  3968 impressions →  283 bookings →  7.13%
  Position  4:  3951 impressions →  224 bookings →  5.67%
  Position  5:   177 i

## Task 3 Interpretation (Your Analysis)

Run the code above and answer these questions:

1. **What is the click rate at position 1 vs position 10?** What's the elasticity (ratio)?
    - Position 1 click rate: 18.78%
    - Position 10 click rate: 4.15%
    - → Click elasticity (Pos 1 vs 10): 4.53x

2. **What is the booking rate at position 1 vs position 10?** What's the elasticity?
    - Position 1 booking rate: 13.37%
    - Position 10 booking rate: 2.56%
    - → Booking elasticity (Pos 1 vs 10): 5.22x

3. **Why is the click elasticity larger than the booking elasticity?** What does this tell you?
    - Answer: this means that position has a bigger impact on booking than it does on clicks.

4. **Does this support the hypothesis that "position matters for getting clicks but not for converting clicks to bookings"?**
    - Answer: the high elasticity shows higher position leads to higher clicks and bookings, so ranking really matters, even more so for bookings

*[Discuss your observations with Claude before proceeding to Task 4]*

● Perfect. You've nailed it now.
                                                      
  The key insight: Position matters MORE for bookings than for clicks.
  This is the opposite of what the original hypothesis predicted.      

  This is a PM insight: Position isn't just about discoverability.     
  Position 1 hotels convert at a higher rate (13.37% booking rate) than
   position 10 hotels (2.56%). This suggests position 1 gets hotels    
  that are:
  1. More appealing (people click them)
  2. More trustworthy OR more satisfying (people book them at higher   
  rates)
                                          
  What this tells us (PM lens):                    
  - Position 1 gets better quality hotels (or better-matched hotels)
  - Position 10 gets worse-quality OR worse-matched hotels
  - The algorithm is doing something right—it's not just sorting by
  popularity, it's putting genuinely bookable hotels at the top
                                                
  This connects back to Task 2: We saw click rates were flat across
  segments (4-5%), but booking rates varied by quality trust. Here we
  see position affects both—and position 1 hotels have better quality
  alignment (or Expedia's algorithm is matching position 1 hotels to
  user intent better).                                                    
  You're ready for Part 5: Competitive Positioning.                    
  
  This will answer: Does being underpriced (comp_score > 0) actually
  drive bookings? Or is the 0.0% median real—price doesn't matter?

---

## Part 5: Analysis 3 - Competitive Positioning

**Question:** Does price advantage (being underpriced) drive bookings? Or is price not a booking lever?

**Hypothesis:** If price drives bookings, underpriced hotels should have higher booking rates than overpriced hotels.

In [14]:
print("\n✓ COMPETITIVE POSITIONING ANALYSIS")
print("=" * 80)

# Filter to only hotels with comp_score data (not NaN)
df_with_comp = df[df['comp_score'].notna()].copy()
print(f"\nHotels with competitor pricing data: {len(df_with_comp):,} ({len(df_with_comp)/len(df)*100:.1f}%)")

# Split data into three groups based on pricing position
# Use ranges to handle floating point precision
underpriced = df_with_comp[df_with_comp['comp_score'] > 0.5].copy()
at_parity = df_with_comp[(df_with_comp['comp_score'] >= -0.5) & (df_with_comp['comp_score'] <= 0.5)].copy()
overpriced = df_with_comp[df_with_comp['comp_score'] < -0.5].copy()

print("\nBreakdown by Pricing Position:")
print("-" * 80)

pricing_groups = [
    ('Underpriced (comp_score > 0.5%)', underpriced),
    ('At Parity (comp_score -0.5% to 0.5%)', at_parity),
    ('Overpriced (comp_score < -0.5%)', overpriced)
]

results = []
for label, group_data in pricing_groups:
    hotel_count = len(group_data)
    click_rate = group_data['click_bool'].mean() * 100 if len(group_data) > 0 else 0
    booking_rate = group_data['booking_bool'].mean() * 100 if len(group_data) > 0 else 0
    
    results.append({
        'Group': label,
        'Hotels': hotel_count,
        'Pct_of_Total': f"{hotel_count/len(df_with_comp)*100:.1f}%",
        'Click_Rate': f"{click_rate:.2f}%",
        'Booking_Rate': f"{booking_rate:.2f}%"
    })
    
    print(f"\n  {label}:")
    print(f"    Hotels: {hotel_count:,} ({hotel_count/len(df_with_comp)*100:.1f}% of comp data)")
    print(f"    Click rate: {click_rate:.2f}%")
    print(f"    Booking rate: {booking_rate:.2f}%")

# Calculate correlation between comp_score and booking
correlation = df_with_comp['comp_score'].corr(df_with_comp['booking_bool'])

print(f"\n\nCorrelation Analysis:")
print("-" * 80)
print(f"  Correlation (comp_score → booking): {correlation:+.4f}")

if abs(correlation) < 0.05:
    strength = "negligible"
    meaning = "price position has almost NO effect on bookings"
elif abs(correlation) < 0.2:
    strength = "weak"
    meaning = "price has minimal impact on conversion"
else:
    strength = "moderate/strong"
    meaning = "price has meaningful impact on conversion"

print(f"  Strength: {strength}")
print(f"  Interpretation: {meaning}")


✓ COMPETITIVE POSITIONING ANALYSIS

Hotels with competitor pricing data: 31,285 (31.4%)

Breakdown by Pricing Position:
--------------------------------------------------------------------------------

  Underpriced (comp_score > 0.5%):
    Hotels: 14,500 (46.3% of comp data)
    Click rate: 5.10%
    Booking rate: 3.38%

  At Parity (comp_score -0.5% to 0.5%):
    Hotels: 6,409 (20.5% of comp data)
    Click rate: 4.60%
    Booking rate: 3.01%

  Overpriced (comp_score < -0.5%):
    Hotels: 10,376 (33.2% of comp data)
    Click rate: 3.82%
    Booking rate: 2.55%


Correlation Analysis:
--------------------------------------------------------------------------------
  Correlation (comp_score → booking): +0.0289
  Strength: negligible
  Interpretation: price position has almost NO effect on bookings


## Task 4 Interpretation (Your Analysis)

Run the code above and answer these questions:

1. **Do underpriced hotels have higher booking rates than overpriced hotels?** By how much? yes, but a very small difference of <1pp

2. **Is the difference statistically or practically significant?** Not really

3. **What does the correlation value tell you about the relationship between price and bookings?** The correlation value is close to 0 so there's no statistically significant impact

4. **Does this support or contradict the earlier finding that "price is not the conversion lever"?** This supports the finding

*[Discuss your observations with Claude before proceeding to Task 5]*

● Solid interpretation. You're exactly right on the conclusion—but let me add a PM nuance:
What the data actually shows:
| Pricing Position | Booking Rate | vs Overprice |
|-----------|---------|--------|
| Underpriced | 3.38% | +32% lift |
| At Parity | 3.01% | +18% lift|
| Overpriced| 2.55% | baseline |

  There IS a trend: underpriced hotels book 32% more than overpriced ones. But the absolute difference is tiny (0.83pp on a 2.55% base), and the   correlation is negligible (+0.0289).So the nuance is: Price has a real but negligible effect on bookings.

  Compare this to what we found:
  - Position 1 vs 10: 5.22x booking elasticity (position matters A LOT)
  - Quality gap (Budget vs Luxury): 37% difference in booking rate (quality trust matters A LOT)
  - Price (underpriced vs overpriced): 32% difference but correlation ~0 (price matters almost not at all)

  The PM insight: If we had a choice between:
  - Making a hotel cheaper (32% lift)
  - Improving its ranking position (522% lift)
  - Fixing its quality-promise alignment (hundreds of % of impact)

  ...we should pick position or quality. Price optimization is the wrong lever.

  Ready for Part 6: Quality Trust Gap Deep-Dive? This is where the real story is.

---

## Part 6: Analysis 4 - Pricing Dynamics

**Question:** What is the price elasticity of demand? Does absolute price affect bookings?

**Hypothesis:** If price is a booking driver, we should see lower booking rates at higher price points. If not, booking rates should be flat across price quartiles.

In [ ]:
print("\n✓ PRICING DYNAMICS ANALYSIS")
print("=" * 80)

# Create price quartiles and calculate booking rate for each
# TODO: Use pd.qcut to create 4 price quartiles
# TODO: For each quartile, calculate:
#   - Average price in that quartile
#   - Booking rate

print("\nBooking Rate by Price Quartile:")
print(f"  Q1 (Cheapest):  Avg ${0:.0f} → {0:.2f}% booking rate")
print(f"  Q2:             Avg ${0:.0f} → {0:.2f}% booking rate")
print(f"  Q3:             Avg ${0:.0f} → {0:.2f}% booking rate")
print(f"  Q4 (Most Expensive): Avg ${0:.0f} → {0:.2f}% booking rate")

print(f"\nPrice Elasticity:")
# TODO: Calculate ratio of booking rates (Q1 vs Q4)
print(f"  Q1 booking rate / Q4 booking rate: [YOUR CALCULATION]x")

## Task 5 Interpretation (Your Analysis)

Run the code above and answer these questions:

1. **Do booking rates differ significantly across price quartiles?** How much?

2. **Is there a clear downward trend as price increases?** Or are rates flat?

3. **What is the price elasticity (ratio of Q1 booking rate to Q4 booking rate)?**

4. **Does this suggest price is a major booking driver, or something else?**

*[Discuss your observations with Claude before proceeding to Task 6]*

In [ ]:
print("\n✓ QUALITY TRUST GAP DEEP-DIVE")
print("=" * 80)

# Calculate quality gap for each segment
# quality_gap = (actual_review_score - promised_star_rating) / promised_star_rating * 100

print("\nSegment-Level Quality Analysis:")
print("-" * 80)

for segment in ['Budget', 'Mid', 'Luxury']:
    seg_data = df[df['market_segment'] == segment]
    
    # Calculate quality metrics
    avg_rating = seg_data['prop_starrating'].mean()
    avg_review = seg_data['prop_review_score'].mean()
    quality_gap = ((avg_review - avg_rating) / avg_rating * 100) if avg_rating > 0 else 0
    booking_rate = seg_data['booking_bool'].mean() * 100
    
    print(f"\n{segment}:")
    print(f"  Promised (Star Rating): {avg_rating:.2f}★")
    print(f"  Actual (Review Score): {avg_review:.2f}")
    print(f"  Quality Gap: {quality_gap:+.1f}%")
    print(f"  Booking Rate: {booking_rate:.2f}%")

# Create a breakdown by "beat promise" vs "fail promise"
# Beat promise: review_score > star_rating
# Fail promise: review_score < star_rating

print("\n\nBeat Promise vs Fail Promise:")
print("-" * 80)

for segment in ['Budget', 'Mid', 'Luxury']:
    seg_data = df[df['market_segment'] == segment]
    
    # Split into beat promise and fail promise
    beat_promise = seg_data[seg_data['prop_review_score'] > seg_data['prop_starrating']]
    fail_promise = seg_data[seg_data['prop_review_score'] < seg_data['prop_starrating']]
    
    # Calculate metrics for each group
    beat_count = len(beat_promise)
    beat_booking_rate = beat_promise['booking_bool'].mean() * 100 if len(beat_promise) > 0 else 0
    
    fail_count = len(fail_promise)
    fail_booking_rate = fail_promise['booking_bool'].mean() * 100 if len(fail_promise) > 0 else 0
    
    # Calculate percentage of hotels in each group
    beat_pct = (beat_count / len(seg_data) * 100) if len(seg_data) > 0 else 0
    fail_pct = (fail_count / len(seg_data) * 100) if len(seg_data) > 0 else 0
    
    print(f"\n{segment}:")
    print(f"  Beat Promise: {beat_count:,} hotels ({beat_pct:.0f}%) → {beat_booking_rate:.2f}% booking rate")
    print(f"  Fail Promise: {fail_count:,} hotels ({fail_pct:.0f}%) → {fail_booking_rate:.2f}% booking rate")
    if fail_count > 0 and beat_count > 0:
        booking_lift = (beat_booking_rate / fail_booking_rate) if fail_booking_rate > 0 else 0
        print(f"  → Lift (Beat vs Fail): {booking_lift:.2f}x")

## Task 6 Interpretation (Your Analysis)

Run the code above and answer these questions:

1. **Which segment has the biggest gap between promised and actual quality?** Is it positive (beats) or negative (fails)?

2. **Which segment has the most hotels that fail their promise?** What's the booking rate for those hotels?

3. **Do hotels that beat their promise have higher booking rates than hotels that fail their promise?** By how much?

4. **Why does the luxury segment have a negative quality gap despite having the highest star rating?** What does this mean for conversion?

*[Discuss your observations with Claude before proceeding to Task 7]*

---

## Part 8: Analysis 6 - User Behavior Segmentation

**Question:** Do different types of users convert at different rates?

**Hypothesis:** Returning visitors know what they want and convert more often. New visitors are more exploratory and may have lower conversion rates.

In [ ]:
print("\n✓ USER BEHAVIOR SEGMENTATION")
print("=" * 80)

# Segment users by whether they are new or returning
# TODO: Identify the column that indicates visitor type (returning vs new)
# TODO: For each visitor type, calculate:
#   - Number of searches
#   - Click rate
#   - Booking rate
#   - Average booking value

print("\nNew Visitors:")
# TODO: Filter for new visitors
print(f"  Searches: [YOUR COUNT]")
print(f"  Click Rate: [YOUR RATE]%")
print(f"  Booking Rate: [YOUR RATE]%")
print(f"  Avg Booking Value: $[YOUR VALUE]")

print("\nReturning Visitors:")
# TODO: Filter for returning visitors
print(f"  Searches: [YOUR COUNT]")
print(f"  Click Rate: [YOUR RATE]%")
print(f"  Booking Rate: [YOUR RATE]%")
print(f"  Avg Booking Value: $[YOUR VALUE]")

print("\nComparison:")
# TODO: Calculate conversion lift
print(f"  Returning visitor booking lift: [YOUR CALCULATION]x")

## Task 7 Interpretation (Your Analysis)

Run the code above and answer these questions:

1. **Do new and returning visitors have different click rates?** If so, why might that be?

2. **Do new and returning visitors have different booking rates?** Which group converts more?

3. **What is the booking rate lift for returning visitors compared to new visitors?** (e.g., 2x, 3x, etc.)

4. **What does this tell you about how experience affects conversion?** Is it about trust, familiarity, or something else?

*[Discuss your observations with Claude before proceeding to the PowerPoint deck]*

---

## Next: Create PowerPoint Deck

After you've completed all analyses (Tasks 2-7) and discussed findings with Claude:

**Task 8 - Consultant Deck (PowerPoint)**
- Create: `presentations/Expedia-Marketplace-Analysis.pptx`
- 12-15 slides with:
  - Problem statement & methodology
  - Key findings (1-6) with data visualizations
  - Strategic recommendations
  - Implementation roadmap
  - Success metrics

The deck will be built from the insights you've discovered in this notebook.